# Train a downstream head on Rem3Di descriptors

Fit a predictor on frozen descriptors + your labels, then score it. This notebook uses **synthetic** `X`/`y` so it runs anywhere (no model or GPU) — swap in real descriptors from `01_get_descriptors.ipynb` and your own targets.

See [docs/train-downstream.md](../docs/train-downstream.md).

In [ ]:
import numpy as np

# Stand-in for real descriptors (X) and labels (y).
rng = np.random.default_rng(0)
N, D = 500, 128
X = rng.normal(size=(N, D)).astype(np.float32)
y = (X[:, :5].sum(axis=1) + rng.normal(scale=0.1, size=N)).astype(np.float32)

idx = rng.permutation(N)
tr, va, te = np.split(idx, [int(0.6 * N), int(0.8 * N)])

In [ ]:
from remedi.data_handling.benchmarks import EvalMetric
from remedi.evaluation.benchmark.learners import LinearLearnerConfig
from remedi.evaluation.benchmark.metrics import metric_for

learner = LinearLearnerConfig(ridge_alpha=1.0).build()
y_pred = learner.fit_predict_regression(X[tr], y[tr], X[va], y[va], X[te], seed=0)

rmse = metric_for(EvalMetric.rmse)(y[te], y_pred)
r2 = metric_for(EvalMetric.r2)(y[te], y_pred)
print(f"RMSE: {rmse:.4f}   R2: {r2:.4f}")

Swap the head by swapping the config — same `fit_predict_*` interface:

```python
from remedi.evaluation.benchmark.learners import LightGBMLearnerConfig, MlpLearnerConfig
learner = LightGBMLearnerConfig(learning_rate=0.03, num_leaves=31).build()
learner = MlpLearnerConfig(hidden_dims=[256, 128], dropout=0.2, device="cuda").build()
```